# Notebook 04 — Overthinking on Trivial Problems

**Chapter**: [Chapter 6 — Overthinking and Optimal Length](../chapters/06-overthinking-and-optimal-length.md).

**Claim demonstrated**: A small reasoning-trained model (R1-distilled) emits long chains on trivially easy problems and gets a measurable fraction wrong as a result. Its non-reasoning sibling (base instruction-tuned) gets the same easy problems right with much shorter chains. Reproduces Chen et al. (2024).

**Hardware**: ≥ 8 GB GPU. CPU works (slow).

---

## Setup

Compare two models on a battery of trivial questions:
- A reasoning model: `deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B`.
- Its base sibling: `Qwen/Qwen2.5-Math-1.5B-Instruct`.

Battery: single-step arithmetic, capital-city lookups, definitional questions where chain-of-thought has no expected lift.

In [ ]:
REASONER = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
BASELINE = "Qwen/Qwen2.5-Math-1.5B-Instruct"
MAX_TOKENS = 1024

import re, json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"

TRIVIAL = [
    ("What is 2 + 3?", "5"),
    ("What is 7 - 4?", "3"),
    ("What is 6 \u00d7 8?", "48"),
    ("What is 100 - 1?", "99"),
    ("What is 12 / 4?", "3"),
    ("What is 5 + 5?", "10"),
    ("What is the smallest prime number?", "2"),
    ("What is the square root of 49?", "7"),
    ("How many sides does a triangle have?", "3"),
    ("What is 10 squared?", "100"),
]

def load(name):
    tok = AutoTokenizer.from_pretrained(name)
    mdl = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.bfloat16, device_map=device)
    mdl.eval()
    return tok, mdl

def run(model_name, tok, mdl, question, max_new=MAX_TOKENS):
    prompt = f"Question: {question}\nAnswer with the final number only.\n"
    inputs = tok(prompt, return_tensors="pt").to(device)
    with torch.inference_mode():
        out = mdl.generate(**inputs, max_new_tokens=max_new, do_sample=False, pad_token_id=tok.eos_token_id)
    gen = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return gen

In [ ]:
tok_r, mdl_r = load(REASONER)
tok_b, mdl_b = load(BASELINE)

def find_numeric_answer(text, gold):
    nums = re.findall(r"-?\d+", text)
    return gold in nums

results = []
for q, gold in TRIVIAL:
    r_out = run("reasoner", tok_r, mdl_r, q)
    b_out = run("baseline", tok_b, mdl_b, q)
    results.append({
        "q": q,
        "gold": gold,
        "reasoner_tokens": len(tok_r.encode(r_out)),
        "reasoner_correct": find_numeric_answer(r_out, gold),
        "baseline_tokens": len(tok_b.encode(b_out)),
        "baseline_correct": find_numeric_answer(b_out, gold),
    })

for r in results: print(r)

In [ ]:
# Aggregate
import statistics as st
r_tokens = [r["reasoner_tokens"] for r in results]
b_tokens = [r["baseline_tokens"] for r in results]
r_acc = sum(r["reasoner_correct"] for r in results) / len(results)
b_acc = sum(r["baseline_correct"] for r in results) / len(results)

print(f"Reasoner: mean tokens = {st.mean(r_tokens):.1f}, accuracy = {r_acc:.2f}")
print(f"Baseline: mean tokens = {st.mean(b_tokens):.1f}, accuracy = {b_acc:.2f}")
print(f"Token ratio reasoner/baseline: {st.mean(r_tokens)/max(1, st.mean(b_tokens)):.1f}x")

## Interpretation

Expected qualitative pattern:
- Reasoner emits ~ 10× more tokens than baseline on trivial questions.
- Reasoner gets ~ 70-90% correct on trivials; baseline ~ 95-100%.
- A few of the reasoner failures are explicit overthinking: it arrives at the correct answer mid-chain and then "reconsiders" into a wrong one.

Inspect the reasoner's failure cases to see this pattern in action.

## Inverse demonstration: CoT helps on harder problems

Repeat with multi-step arithmetic (e.g. `'What is 7 * 13 + 8?'`) — expect the reasoner to *win* there even with more tokens. This is the calibrated-difficulty regime that Yang et al. (2025) target with thinking-optimal scaling.